# Assignment 2. ASR Decoding

In [1]:
import os
import csv
import time
import torch
import torchaudio
import jiwer
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

from wav2vec2decoder import Wav2Vec2Decoder

/home/rkharkovskoy/.pyenv/versions/kenlm-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_dataset(data_dir: str):
    csv_path = os.path.join(data_dir, "manifest.csv")
    samples = []
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            audio_path = row["path"]
            if not os.path.isabs(audio_path):
                audio_path = os.path.join(os.getcwd(), audio_path)
            samples.append((audio_path, row["text"].strip().lower()))
    return samples


ls_data = load_dataset("data/librispeech_test_other")
earn_data = load_dataset("data/earnings22_test")

print(f"LibriSpeech samples: {len(ls_data)}")
print(f"Earnings22 samples: {len(earn_data)}")

LibriSpeech samples: 200
Earnings22 samples: 200


In [3]:
print("LibriSpeech examples:")
for path, text in ls_data[:3]:
    print(f"{os.path.basename(path)}: {text[:80]}")

print("Earnings22 examples:")
for path, text in earn_data[:3]:
    print(f"{os.path.basename(path)}: {text[:80]}")

LibriSpeech examples:
sample_0.wav: i am from the cutter lying off the coast
sample_1.wav: don't cry he said i was obliged to come
sample_2.wav: and and you have not found out anything came in quick frightened tones
Earnings22 examples:
sample_000000.wav: but happy with what we've been seeing here
sample_000001.wav: i just just curious to get your sense on on kind of visibility of of this busine
sample_000002.wav: you also saw that we have a new leader lee she's joined us uh as the new preside


## Общая функция оценки

Прогоняем весь датасет через декодер, считаем WER и CER. `max_samples` нужен для быстрых прогонов во время отладки.

In [4]:
def evaluate(decoder, dataset, method="greedy", max_samples=None):
    refs = []
    hyps = []
    for i, (audio_path, ref) in enumerate(dataset):
        if max_samples and i >= max_samples:
            break
        audio, sr = torchaudio.load(audio_path)
        if sr != 16000:
            audio = torchaudio.transforms.Resample(sr, 16000)(audio)
        hyp = decoder.decode(audio, method=method)
        refs.append(ref)
        hyps.append(hyp)
    wer = jiwer.wer(refs, hyps)
    cer = jiwer.cer(refs, hyps)
    return wer, cer, list(zip(refs, hyps))

### Task 1. Implement greedy_decode

Тут на каждом шаге берём токен с максимальной вероятностью, потом коллапсируем повторы и убираем blank. LM не используется.

In [5]:
def task1(dataset):
    decoder = Wav2Vec2Decoder(lm_model_path=None)
    wer, cer, _ = evaluate(decoder, dataset, method="greedy")
    print(f"WER={wer*100:.2f}%, CER={cer*100:.2f}%")
    return wer, cer


t1_wer, t1_cer = task1(ls_data)

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 42181.80it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


WER=11.22%, CER=3.81%


### Task 2. Implement beam_search_decode

Проверка разной ширины луча (1, 3, 10, 50)

In [6]:
def task2(dataset):
    beam_widths = [1, 3, 10, 50]
    results = {}

    for bw in beam_widths:
        t0 = time.time()
        decoder = Wav2Vec2Decoder(lm_model_path=None, beam_width=bw)
        wer, cer, _ = evaluate(decoder, dataset, method="beam")
        elapsed = time.time() - t0
        print(f"beam_width={bw}, WER={wer*100:.2f}%, CER={cer*100:.2f}%, time={elapsed:.1f}s")
        results[bw] = {"wer": wer, "cer": cer, "time": elapsed}

    fig, ax1 = plt.subplots(figsize=(8, 5))
    bws = list(results.keys())
    wers = [results[b]["wer"] * 100 for b in bws]
    times = [results[b]["time"] for b in bws]

    ax1.set_xlabel("Beam Width")
    ax1.set_ylabel("WER (%)", color="tab:blue")
    ax1.plot(bws, wers, "o-", color="tab:blue")
    ax1.tick_params(axis="y", labelcolor="tab:blue")

    ax2 = ax1.twinx()
    ax2.set_ylabel("Time (s)", color="tab:red")
    ax2.plot(bws, times, "s--", color="tab:red")
    ax2.tick_params(axis="y", labelcolor="tab:red")

    plt.title("Beam Width vs WER and Compute Time")
    fig.tight_layout()
    plt.savefig("plots/task2_beam_width.png", dpi=150)
    plt.show()

    return results

t2_results = task2(ls_data)

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 35338.70it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


beam_width=1, WER=11.24%, CER=3.80%, time=53.7s


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 34848.43it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


beam_width=3, WER=11.15%, CER=3.78%, time=67.8s


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 32645.29it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


beam_width=10, WER=11.07%, CER=3.77%, time=110.9s


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 32465.31it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


beam_width=50, WER=11.10%, CER=3.77%, time=447.0s


/tmp/ipykernel_371524/3727610831.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


При beam_width=1 beam search эквивалентен greedy WER должен совпадать. Дальше WER снижается, но после определённого момента прирост становится минимальным, а время растёт существенно. Оптимальным компромиссом выглядит beam_width=10.

### Task 3. Implement temperature scaling for acoustic model outputs.

In [17]:
def task3(dataset):
    temps = [0.5, 0.8, 1.0, 1.2, 1.5, 2.0]
    results = {}

    for T in temps:
        decoder = Wav2Vec2Decoder(lm_model_path=None, temperature=T)
        wer, cer, _ = evaluate(decoder, dataset, method="greedy")
        print(f"T={T}, WER={wer*100:.2f}%, CER={cer*100:.2f}%")
        results[T] = {"wer": wer, "cer": cer}

    ts = list(results.keys())
    plt.figure(figsize=(8, 5))
    plt.plot(ts, [results[t]["wer"] * 100 for t in ts], "o-")
    plt.xlabel("Temperature")
    plt.ylabel("WER (%)")
    plt.title("Temperature vs WER (greedy, LibriSpeech)")
    plt.grid(True, alpha=0.3)
    plt.savefig("plots/task3_temperature.png", dpi=150)
    plt.show()

    return results

t3_results = task3(ls_data)

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 39994.26it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


T=0.5, WER=11.22%, CER=3.81%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 40582.01it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


T=0.8, WER=11.22%, CER=3.81%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 37921.89it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 40028.47it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


T=1.5, WER=11.22%, CER=3.81%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 32775.25it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


T=2.0, WER=11.22%, CER=3.81%


/tmp/ipykernel_371524/2911941687.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Task 4. Implement beam_search_with_lm - shallow fusion of the provided 3-gram LM.

In [8]:
def task4(dataset):
    alphas = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0]
    betas = [0.0, 0.5, 1.0, 1.5]

    results = {}
    best_wer = 1.0
    best_config = (0, 0)

    for alpha in alphas:
        for beta in betas:
            decoder = Wav2Vec2Decoder(
                lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
                beam_width=10,
                alpha=alpha,
                beta=beta
            )
            wer, cer, _ = evaluate(decoder, dataset, method="beam_lm")
            print(f"alpha={alpha}, beta={beta}, WER={wer*100:.2f}%, CER={cer*100:.2f}%")
            results[(alpha, beta)] = {"wer": wer, "cer": cer}
            if wer < best_wer:
                best_wer = wer
                best_config = (alpha, beta)

    print(f"best: alpha={best_config[0]}, beta={best_config[1]}, WER={best_wer*100:.2f}%")

    wer_matrix = np.zeros((len(alphas), len(betas)))
    for i, a in enumerate(alphas):
        for j, b in enumerate(betas):
            wer_matrix[i, j] = results[(a, b)]["wer"] * 100

    plt.figure(figsize=(10, 7))
    sns.heatmap(wer_matrix, annot=True, fmt=".2f",
                xticklabels=[str(b) for b in betas],
                yticklabels=[str(a) for a in alphas],
                cmap="YlOrRd_r")
    plt.xlabel("Beta")
    plt.ylabel("Alpha")
    plt.title("WER (%) - Shallow Fusion alpha/beta sweep")
    plt.tight_layout()
    plt.savefig("plots/task4_heatmap.png", dpi=150)
    plt.show()

    return results, best_config

t4_results, best_sf_config = task4(ls_data)
print(f"Best sf config: alpha={best_sf_config[0]}, beta={best_sf_config[1]}")

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 29479.58it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


alpha=0.01, beta=0.0, WER=11.07%, CER=3.77%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 41879.83it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


alpha=0.01, beta=0.5, WER=11.02%, CER=3.75%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 35719.15it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


...

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 33135.55it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


alpha=5.0, beta=1.0, WER=99.98%, CER=22.48%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 32428.61it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


alpha=5.0, beta=1.5, WER=99.98%, CER=22.48%
best: alpha=0.1, beta=0.5, WER=10.98%
Best sf config: alpha=0.1, beta=0.5


/tmp/ipykernel_371524/383744957.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Task 5: 3-gram and 4-gram LM 

Сравниваем качество с 3-граммовой pruned и 4-граммовой языковыми моделями, используя найденные выше оптимальные alpha и beta.

In [9]:
def task5(dataset, best_alpha, best_beta):
    lm4_path = "lm/4-gram.arpa.gz"

    decoder_3g = Wav2Vec2Decoder(
        lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
        beam_width=10,
        alpha=best_alpha,
        beta=best_beta
    )
    wer3, cer3, _ = evaluate(decoder_3g, dataset, method="beam_lm")

    decoder_4g = Wav2Vec2Decoder(
        lm_model_path=lm4_path,
        beam_width=10,
        alpha=best_alpha,
        beta=best_beta
    )
    wer4, cer4, _ = evaluate(decoder_4g, dataset, method="beam_lm")

    print(f"3gram WER={wer3*100:.2f}%, CER={cer3*100:.2f}%")
    print(f"4gram WER={wer4*100:.2f}%, CER={cer4*100:.2f}%")

    return {
        "3gram": {
            "wer": wer3, 
            "cer": cer3
        }, 
        "4gram": {
            "wer": wer4,
            "cer": cer4
        }
    }

t5_results = task5(ls_data, best_sf_config[0], best_sf_config[1])

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 19227.03it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************
Loading weights: 100%|██████████| 212/212 [00:00<00:00, 24724.51it/s]
Wav2Vec2ForC

3gram WER=10.98%, CER=3.74%
4gram WER=11.02%, CER=3.75%


### Task 6. Implement lm_rescore - second-pass LM rescoring of beam hypotheses.

На втором проходе сначала получаем топ-N гипотез через beam search (без LM), потом перевзвешиваем их с помощью LM

In [11]:
def task6(dataset, best_sf_alpha, best_sf_beta):
    alphas = [0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0]
    betas = [0.0, 0.5, 1.0, 1.5]

    results = {}
    best_wer = 1.0
    best_config = (0, 0)

    for alpha in alphas:
        for beta in betas:
            decoder = Wav2Vec2Decoder(
                lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
                beam_width=10,
                alpha=alpha,
                beta=beta
            )
            wer, cer, _ = evaluate(decoder, dataset, method="beam_lm_rescore")
            print(f"alpha={alpha}, beta={beta}, WER={wer*100:.2f}%, CER={cer*100:.2f}%")
            results[(alpha, beta)] = {"wer": wer, "cer": cer}
            if wer < best_wer:
                best_wer = wer
                best_config = (alpha, beta)

    print(f"best: alpha={best_config[0]}, beta={best_config[1]}, WER={best_wer*100:.2f}%")

    wer_matrix = np.zeros((len(alphas), len(betas)))
    for i, a in enumerate(alphas):
        for j, b in enumerate(betas):
            wer_matrix[i, j] = results[(a, b)]["wer"] * 100

    plt.figure(figsize=(10, 7))
    sns.heatmap(wer_matrix, annot=True, fmt=".2f",
                xticklabels=[str(b) for b in betas],
                yticklabels=[str(a) for a in alphas],
                cmap="YlOrRd_r")
    plt.xlabel("Beta")
    plt.ylabel("Alpha")
    plt.title("WER (%) — Rescoring alpha/beta sweep")
    plt.tight_layout()
    plt.savefig("plots/task6_heatmap.png", dpi=150)
    plt.show()

    decoder_beam = Wav2Vec2Decoder(lm_model_path=None, beam_width=10)
    decoder_sf = Wav2Vec2Decoder(
        lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
        beam_width=10,
        alpha=best_sf_alpha,
        beta=best_sf_beta
    )
    decoder_rs = Wav2Vec2Decoder(
        lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
        beam_width=10,
        alpha=best_config[0],
        beta=best_config[1]
    )

    comparisons = []
    for audio_path, ref in dataset[:50]:
        audio, sr = torchaudio.load(audio_path)
        if sr != 16000:
            audio = torchaudio.transforms.Resample(sr, 16000)(audio)
        hyp_beam = decoder_beam.decode(audio, method="beam")
        hyp_sf = decoder_sf.decode(audio, method="beam_lm")
        hyp_rs = decoder_rs.decode(audio, method="beam_lm_rescore")

        if hyp_beam != hyp_sf or hyp_beam != hyp_rs:
            comparisons.append({"ref": ref, "beam": hyp_beam, "sf": hyp_sf, "rs": hyp_rs})

        if len(comparisons) >= 10:
            break

    with open("task6_qualitative.txt", "w") as f:
        for c in comparisons:
            mark_sf = "correct" if jiwer.wer(c["ref"], c["sf"]) <= jiwer.wer(c["ref"], c["beam"]) else "worse"
            mark_rs = "correct" if jiwer.wer(c["ref"], c["rs"]) <= jiwer.wer(c["ref"], c["beam"]) else "worse"
            f.write(f"REF: {c['ref']}\n")
            f.write(f"BEAM: {c['beam']}\n")
            f.write(f"SF: {c['sf']} ({mark_sf})\n")
            f.write(f"RS: {c['rs']} ({mark_rs})\n\n")

    print(f"({len(comparisons)} diffs found)")
    return results, best_config


t6_results, best_rs_config = task6(ls_data, best_sf_config[0], best_sf_config[1])
print(f"Best rs config: alpha={best_rs_config[0]}, beta={best_rs_config[1]}")

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 24813.52it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


alpha=0.01, beta=0.0, WER=11.07%, CER=3.77%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 25443.30it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


alpha=0.01, beta=0.5, WER=11.02%, CER=3.75%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 22990.81it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


...

alpha=5.0, beta=1.0, WER=12.81%, CER=3.99%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 26288.80it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


alpha=5.0, beta=1.5, WER=12.69%, CER=3.97%
best: alpha=0.01, beta=1.5, WER=11.00%


/tmp/ipykernel_371524/2789173063.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
Loading weights: 100%|██████████| 212/212 [00:00<00:00, 24566.72it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 212/212 [00:00<00:00, 27753.44it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING  

(4 diffs found)
Best rs config: alpha=0.01, beta=1.5


### Task 7. Evaluate your best shallow-fusion and rescoring configurations (from Tasks 4–6) on data/earnings22_test/.

In [12]:
def task7(ls_dataset, earn_dataset, best_sf_config, best_rs_config):
    results_table = {}

    for domain_name, dataset in [("LibriSpeech", ls_dataset), ("Earnings22", earn_dataset)]:
        decoder = Wav2Vec2Decoder(lm_model_path=None)
        wer, cer, _ = evaluate(decoder, dataset, method="greedy")
        results_table[(domain_name, "Greedy")] = (wer, cer)
        print(f"{domain_name} greedy: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

        decoder = Wav2Vec2Decoder(lm_model_path=None, beam_width=10)
        wer, cer, _ = evaluate(decoder, dataset, method="beam")
        results_table[(domain_name, "Beam")] = (wer, cer)
        print(f"{domain_name} beam: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

        decoder = Wav2Vec2Decoder(
            lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
            beam_width=10,
            alpha=best_sf_config[0],
            beta=best_sf_config[1]
        )
        wer, cer, _ = evaluate(decoder, dataset, method="beam_lm")
        results_table[(domain_name, "SF")] = (wer, cer)
        print(f"{domain_name} SF: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

        decoder = Wav2Vec2Decoder(
            lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
            beam_width=10,
            alpha=best_rs_config[0],
            beta=best_rs_config[1]
        )
        wer, cer, _ = evaluate(decoder, dataset, method="beam_lm_rescore")
        results_table[(domain_name, "RS")] = (wer, cer)
        print(f"{domain_name} RS: WER={wer*100:.2f}%, CER={cer*100:.2f}%")

    return results_table

t7_results = task7(ls_data, earn_data, best_sf_config, best_rs_config)

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 21058.44it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LibriSpeech greedy: WER=11.22%, CER=3.81%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 26087.50it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LibriSpeech beam: WER=11.07%, CER=3.77%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 24585.74it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


...

Earnings22 beam: WER=54.94%, CER=25.38%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 20183.23it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


Earnings22 SF: WER=55.57%, CER=25.47%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 27115.31it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


Earnings22 RS: WER=55.33%, CER=25.38%


Сводная таблица

In [14]:
import pandas as pd

rows = [
    {
        "Domain": domain,
        "Method": method,
        "WER (%)": round(wer * 100, 2),
        "CER (%)": round(cer * 100, 2)
    }
    for (domain, method), (wer, cer) in t7_results.items()
]
pd.DataFrame(rows).set_index(["Domain", "Method"])

WER (%)  CER (%)
Domain      Method                  
LibriSpeech Greedy    11.22     3.81
            Beam      11.07     3.77
            SF        10.98     3.74
            RS        11.00     3.74
Earnings22  Greedy    54.97    25.58
            Beam      54.94    25.38
            SF        55.57    25.47
            RS        55.33    25.38

In [15]:
def task7b(earn_dataset, best_sf_config):
    temps = [0.5, 1.0, 1.5, 2.0]
    greedy_wers = []
    sf_wers = []

    for T in temps:
        decoder_g = Wav2Vec2Decoder(lm_model_path=None, temperature=T)
        wer_g, _, _ = evaluate(decoder_g, earn_dataset, method="greedy")
        greedy_wers.append(wer_g * 100)

        decoder_sf = Wav2Vec2Decoder(
            lm_model_path="lm/3-gram.pruned.1e-7.arpa.gz",
            beam_width=10,
            alpha=best_sf_config[0],
            beta=best_sf_config[1],
            temperature=T
        )
        wer_sf, _, _ = evaluate(decoder_sf, earn_dataset, method="beam_lm")
        sf_wers.append(wer_sf * 100)
        print(f"T={T}, greedy={wer_g*100:.2f}%, SF={wer_sf*100:.2f}%")

    plt.figure(figsize=(8, 5))
    plt.plot(temps, greedy_wers, "o-", label="Greedy")
    plt.plot(temps, sf_wers, "s-", label="Shallow Fusion")
    plt.xlabel("Temperature")
    plt.ylabel("WER (%)")
    plt.title("Temperature vs WER (Earnings22)")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig("plots/task7b_temperature_earnings.png", dpi=150)
    plt.show()

    return {"greedy": dict(zip(temps, greedy_wers)), "sf": dict(zip(temps, sf_wers))}

t7b_results = task7b(earn_data, best_sf_config)

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 20595.55it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 212/212 [00:00<00:00, 21783.79it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
-

T=0.5, greedy=54.97%, SF=55.18%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 23646.22it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 212/212 [00:00<00:00, 25256.84it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
-

T=1.0, greedy=54.97%, SF=55.57%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 25883.23it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 212/212 [00:00<00:00, 29982.55it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
-

T=1.5, greedy=54.97%, SF=56.60%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 32724.59it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading weights: 100%|██████████| 212/212 [00:00<00:00, 28075.03it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
-

T=2.0, greedy=54.97%, SF=58.40%


/tmp/ipykernel_371524/3206005472.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Task 8. Train a financial-domain KenLM model using data/earnings22_train/corpus.txt as your base training corpus (~5000 lines of real earnings-call speech).

На моем ноутбуке с арчем не хотело собираться, поэтому собрал в гугл колабе

### Task 9. Apply your two best decoding methods using both available LMs (LibriSpeech 3-gram and your financial-domain LM) on both test sets.

Сравнение LibriSpeech 3-gram LM и финансовую LM на обоих доменах. 

Скорее всего финансовая LM должна помочь на Earnings22 и навредить на LibriSpeech.

In [16]:
def task9(ls_dataset, earn_dataset, best_sf_config, best_rs_config):
    fin_lm = "lm/financial-3gram.arpa.gz"

    results = {}

    for domain_name, dataset in [("LibriSpeech", ls_dataset), ("Earnings22", earn_dataset)]:
        for lm_name, lm_path in [("LibriSpeech-3gram", "lm/3-gram.pruned.1e-7.arpa.gz"),
                                   ("Financial-3gram", fin_lm)]:
            decoder = Wav2Vec2Decoder(
                lm_model_path=lm_path,
                beam_width=10,
                alpha=best_sf_config[0],
                beta=best_sf_config[1]
            )
            wer_sf, cer_sf, _ = evaluate(decoder, dataset, method="beam_lm")
            results[(domain_name, lm_name, "SF")] = (wer_sf, cer_sf)
            print(f"{domain_name} {lm_name} SF: WER={wer_sf*100:.2f}%, CER={cer_sf*100:.2f}%")

            decoder = Wav2Vec2Decoder(
                lm_model_path=lm_path,
                beam_width=10,
                alpha=best_rs_config[0],
                beta=best_rs_config[1]
            )
            wer_rs, cer_rs, _ = evaluate(decoder, dataset, method="beam_lm_rescore")
            results[(domain_name, lm_name, "RS")] = (wer_rs, cer_rs)
            print(f"{domain_name} {lm_name} RS: WER={wer_rs*100:.2f}%, CER={cer_rs*100:.2f}%")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax_idx, (domain_name, dataset) in enumerate([("LibriSpeech", ls_dataset), ("Earnings22", earn_dataset)]):
        ax = axes[ax_idx]
        methods = ["SF", "RS"]
        lms = ["LibriSpeech-3gram", "Financial-3gram"]
        x = np.arange(len(methods))
        width = 0.35

        for i, lm_name in enumerate(lms):
            wers = [results.get((domain_name, lm_name, m), (0, 0))[0] * 100 for m in methods]
            ax.bar(x + i * width, wers, width, label=lm_name)

        ax.set_xlabel("Method")
        ax.set_ylabel("WER (%)")
        ax.set_title(domain_name)
        ax.set_xticks(x + width / 2)
        ax.set_xticklabels(methods)
        ax.legend()

    plt.suptitle("WER by Domain and LM")
    plt.tight_layout()
    plt.savefig("plots/task9_lm_comparison.png", dpi=150)
    plt.show()

    return results

t9_results = task9(ls_data, earn_data, best_sf_config, best_rs_config)

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 30670.27it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


LibriSpeech LibriSpeech-3gram SF: WER=10.98%, CER=3.74%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 26427.88it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/3-gram.pruned.1e-7.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


LibriSpeech LibriSpeech-3gram RS: WER=11.00%, CER=3.74%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 21624.86it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/financial-3gram.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


...

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 25740.12it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/financial-3gram.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


Earnings22 Financial-3gram SF: WER=52.56%, CER=25.12%


Loading weights: 100%|██████████| 212/212 [00:00<00:00, 28492.45it/s]
Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-base-100h
Key                           | Status     | 
------------------------------+------------+-
wav2vec2.mask_time_emb_vector | UNEXPECTED | 
wav2vec2.masked_spec_embed    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Loading the LM will be faster if you build a binary file.
Reading /home/rkharkovskoy/projects/university/speech-course/hw2/lm/financial-3gram.arpa.gz
----5---10---15---20---25---30---35---40---45---50---55---60---65---70---75---80---85---90---95--100
****************************************************************************************************


Earnings22 Financial-3gram RS: WER=55.18%, CER=25.36%


/tmp/ipykernel_371524/3617023482.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
